# Experiment 4: Forward Chaining Inference Engine (Data-Driven Reasoning)

**Roll No.:** U23AI071  

## Aim
To implement a forward chaining inference engine and use it to identify an animal from observed
features, in the manner of the classic **ZOOKEEPER** system.

## Theory

Forward chaining starts from the **available data** and applies rules to derive new facts until either a
goal is reached or no rule can fire. It is *data-driven*: reasoning is triggered by what is known, not by
what is asked. This suits monitoring, configuration and interpretation tasks where many conclusions
may follow from the same data.

Each cycle re-computes the applicable rules against the growing fact base and fires one of them,
adding its conclusion to the facts. The chain of fired rules (here **R1 -> R4 -> R6**) is itself valuable: it is
the raw material for the explanation facility built in Experiment 7. Intermediate conclusions such as
`is mammal` and `is carnivore` show how rule chaining builds abstraction level by level.

**Components of the system**

| Component | Role |
|---|---|
| Knowledge Base (KB) | Set of IF-THEN production rules |
| Working Memory (WM) | Set of facts known at the current instant |
| Match | Compare rule conditions against WM |
| Conflict Set | All applicable rules whose conclusion is not already in WM |
| Conflict Resolution | Select one rule (here: the first in rule order) |
| Act | Fire the rule, assert its conclusion into WM |

## Algorithm

1. Initialise the fact set with the observed features.
2. Repeat: find all rules whose conditions are satisfied and whose conclusion is new.
3. If none exists, stop; otherwise fire the first such rule, add its conclusion, and record its identifier.
4. On termination, report the final fact set, the firing sequence, and the identification.

## Step 1: Rule representation

A rule is stored as `Rule(id, conditions, conclusion)`.

A rule is **applicable** when every condition is satisfied by WM *and* its conclusion is not already in
WM (the second test prevents a rule from firing twice and guarantees termination).

A condition written as `not X` is satisfied when `X` is **absent** from WM (closed-world assumption).
This is needed for the zebra rule in Exercise 1.

In [22]:
class Rule:
    def __init__(self, rid, conditions, conclusion):
        self.rid = rid                    # rule identifier, e.g. "R1"
        self.conditions = list(conditions)  # premises (antecedents)
        self.conclusion = conclusion        # consequent

    def is_applicable(self, wm):
        # every condition must hold in the working memory
        for c in self.conditions:
            if c.startswith("not "):
                if c[4:] in wm:          # negative condition violated
                    return False
            elif c not in wm:            # positive condition not satisfied
                return False
        # the conclusion must be NEW, otherwise the rule is useless
        return self.conclusion not in wm

    def __str__(self):
        return "{:<4} IF {}  THEN  {}".format(
            self.rid + ":", " AND ".join(self.conditions), self.conclusion)


def show_kb(rules, title="KNOWLEDGE BASE"):
    print(title)
    print("=" * 78)
    for r in rules:
        print(r)
    print("=" * 78)

## Step 2: The animal-identification knowledge base

Six production rules of the ZOOKEEPER style. `mammal`, `bird` and `carnivore` are **intermediate
conclusions** (abstraction levels); `cheetah` is a **goal / identification** fact.

In [23]:
KB = [
    Rule("R1", ["has hair"],                                            "mammal"),
    Rule("R2", ["gives milk"],                                          "mammal"),
    Rule("R3", ["has feathers"],                                        "bird"),
    Rule("R4", ["mammal", "eats meat"],                                 "carnivore"),
    Rule("R5", ["mammal", "pointed teeth", "claws", "forward eyes"],    "carnivore"),
    Rule("R6", ["carnivore", "tawny color", "dark spots"],              "cheetah"),
]

show_kb(KB)

KNOWLEDGE BASE
R1:  IF has hair  THEN  mammal
R2:  IF gives milk  THEN  mammal
R3:  IF has feathers  THEN  bird
R4:  IF mammal AND eats meat  THEN  carnivore
R5:  IF mammal AND pointed teeth AND claws AND forward eyes  THEN  carnivore
R6:  IF carnivore AND tawny color AND dark spots  THEN  cheetah


## Step 3: The forward chaining inference engine

Each cycle: **match -> build conflict set -> resolve (first rule) -> fire -> assert**.
The loop halts when the conflict set becomes empty.

In [24]:
# facts that count as a final identification (goal facts)
GOALS = {"cheetah", "tiger", "giraffe", "zebra", "penguin", "ostrich", "albatross"}

# how specific each derivable class is (used when no animal can be identified)
SPECIFICITY = {
    "mammal": 1, "bird": 1,
    "carnivore": 2, "ungulate": 2,
    "cheetah": 3, "tiger": 3, "giraffe": 3, "zebra": 3,
    "penguin": 3, "ostrich": 3, "albatross": 3,
}


def forward_chain(rules, initial_facts, verbose=True):
    # Data-driven forward chaining. Fires the FIRST applicable rule per cycle.
    wm = set(initial_facts)
    firing_sequence = []
    cycle = 0

    if verbose:
        print("Observed features (initial WM):", sorted(wm))
        print("-" * 78)

    while True:
        cycle += 1
        # MATCH: build the conflict set
        conflict_set = [r for r in rules if r.is_applicable(wm)]

        if not conflict_set:
            if verbose:
                print("Cycle {}: conflict set = []  -> no rule can fire, halt".format(cycle))
            break

        # CONFLICT RESOLUTION: first rule in rule order
        selected = conflict_set[0]

        # ACT: assert the conclusion and record the rule identifier
        wm.add(selected.conclusion)
        firing_sequence.append(selected.rid)

        if verbose:
            print("Cycle {}: conflict set = {}".format(cycle, [r.rid for r in conflict_set]))
            print("         fire {} -> assert '{}'".format(selected.rid, selected.conclusion))
            print("         WM = {}".format(sorted(wm)))
            print("-" * 78)

    return wm, firing_sequence


def report(wm, firing_sequence, initial_facts=None):
    print("\nRESULT")
    print("=" * 78)
    if initial_facts is not None:
        print("Initial facts    :", sorted(initial_facts))
    print("Final WM         :", sorted(wm))
    print("Firing sequence  :", " -> ".join(firing_sequence) if firing_sequence else "(no rule fired)")

    identified = sorted(GOALS & wm)
    if identified:
        print("Identification   :", ", ".join(a.upper() for a in identified))
    else:
        derived = [f for f in wm if f in SPECIFICITY]
        if derived:
            best = max(derived, key=lambda f: SPECIFICITY[f])
            print("Identification   : NOT ESTABLISHED from the available facts")
            print("Most specific class established :", best.upper())
        else:
            print("Identification   : NOT ESTABLISHED (no class could be derived)")
    print("=" * 78)

## Step 4: Running the engine on the given observations

Observed features: `has hair`, `eats meat`, `tawny color`, `dark spots`.

Expected chain: **R1 (mammal) -> R4 (carnivore) -> R6 (cheetah)**.

In [25]:
facts = ["has hair", "eats meat", "tawny color", "dark spots"]

wm, seq = forward_chain(KB, facts)
report(wm, seq, facts)

Observed features (initial WM): ['dark spots', 'eats meat', 'has hair', 'tawny color']
------------------------------------------------------------------------------
Cycle 1: conflict set = ['R1']
         fire R1 -> assert 'mammal'
         WM = ['dark spots', 'eats meat', 'has hair', 'mammal', 'tawny color']
------------------------------------------------------------------------------
Cycle 2: conflict set = ['R4']
         fire R4 -> assert 'carnivore'
         WM = ['carnivore', 'dark spots', 'eats meat', 'has hair', 'mammal', 'tawny color']
------------------------------------------------------------------------------
Cycle 3: conflict set = ['R6']
         fire R6 -> assert 'cheetah'
         WM = ['carnivore', 'cheetah', 'dark spots', 'eats meat', 'has hair', 'mammal', 'tawny color']
------------------------------------------------------------------------------
Cycle 4: conflict set = []  -> no rule can fire, halt

RESULT
Initial facts    : ['dark spots', 'eats meat', 'has hair

---
# Exercise 1
**Add rules and facts to identify a tiger, a giraffe (long neck, long legs) and a zebra (black stripes, not carnivore).**

Three identification rules and one new intermediate class `ungulate` (hoofed mammal) are added.
The zebra rule uses the negative condition `not carnivore`.

In [26]:
KB_EXT = KB + [
    Rule("R7",  ["mammal", "has hooves"],                                        "ungulate"),
    Rule("R8",  ["carnivore", "tawny color", "black stripes"],                   "tiger"),
    Rule("R9",  ["ungulate", "long neck", "long legs", "dark spots"],            "giraffe"),
    Rule("R10", ["ungulate", "black stripes", "not carnivore"],                  "zebra"),
]

show_kb(KB_EXT, "EXTENDED KNOWLEDGE BASE")

EXTENDED KNOWLEDGE BASE
R1:  IF has hair  THEN  mammal
R2:  IF gives milk  THEN  mammal
R3:  IF has feathers  THEN  bird
R4:  IF mammal AND eats meat  THEN  carnivore
R5:  IF mammal AND pointed teeth AND claws AND forward eyes  THEN  carnivore
R6:  IF carnivore AND tawny color AND dark spots  THEN  cheetah
R7:  IF mammal AND has hooves  THEN  ungulate
R8:  IF carnivore AND tawny color AND black stripes  THEN  tiger
R9:  IF ungulate AND long neck AND long legs AND dark spots  THEN  giraffe
R10: IF ungulate AND black stripes AND not carnivore  THEN  zebra


In [27]:
test_cases = {
    "TIGER  ": ["has hair", "eats meat", "tawny color", "black stripes"],
    "GIRAFFE": ["gives milk", "has hooves", "long neck", "long legs", "dark spots"],
    "ZEBRA  ": ["has hair", "has hooves", "black stripes"],
}

for name, f in test_cases.items():
    print("\n" + "#" * 78)
    print("# TEST CASE :", name.strip())
    print("#" * 78)
    wm, seq = forward_chain(KB_EXT, f)
    report(wm, seq, f)


##############################################################################
# TEST CASE : TIGER
##############################################################################
Observed features (initial WM): ['black stripes', 'eats meat', 'has hair', 'tawny color']
------------------------------------------------------------------------------
Cycle 1: conflict set = ['R1']
         fire R1 -> assert 'mammal'
         WM = ['black stripes', 'eats meat', 'has hair', 'mammal', 'tawny color']
------------------------------------------------------------------------------
Cycle 2: conflict set = ['R4']
         fire R4 -> assert 'carnivore'
         WM = ['black stripes', 'carnivore', 'eats meat', 'has hair', 'mammal', 'tawny color']
------------------------------------------------------------------------------
Cycle 3: conflict set = ['R8']
         fire R8 -> assert 'tiger'
         WM = ['black stripes', 'carnivore', 'eats meat', 'has hair', 'mammal', 'tawny color', 'tiger']
----------

---
# Exercise 2
**Give the engine facts that satisfy no identification rule and make it report the most specific class it could establish.**

Case A: a carnivore whose colour/marking features are unknown -> stops at `carnivore` (level 2).  
Case B: only `has feathers` -> stops at `bird` (level 1).  
Case C: features matching nothing at all -> no class can be derived.

In [28]:
incomplete_cases = {
    "A": ["gives milk", "pointed teeth", "claws", "forward eyes"],
    "B": ["has feathers"],
    "C": ["lays eggs", "lives in water"],
}

for name, f in incomplete_cases.items():
    print("\n" + "#" * 78)
    print("# CASE", name)
    print("#" * 78)
    wm, seq = forward_chain(KB_EXT, f)
    report(wm, seq, f)


##############################################################################
# CASE A
##############################################################################
Observed features (initial WM): ['claws', 'forward eyes', 'gives milk', 'pointed teeth']
------------------------------------------------------------------------------
Cycle 1: conflict set = ['R2']
         fire R2 -> assert 'mammal'
         WM = ['claws', 'forward eyes', 'gives milk', 'mammal', 'pointed teeth']
------------------------------------------------------------------------------
Cycle 2: conflict set = ['R5']
         fire R5 -> assert 'carnivore'
         WM = ['carnivore', 'claws', 'forward eyes', 'gives milk', 'mammal', 'pointed teeth']
------------------------------------------------------------------------------
Cycle 3: conflict set = []  -> no rule can fire, halt

RESULT
Initial facts    : ['claws', 'forward eyes', 'gives milk', 'pointed teeth']
Final WM         : ['carnivore', 'claws', 'forward eyes'

---
# Exercise 3
**Modify the engine to fire ALL applicable rules per cycle instead of the first one; compare the firing sequences.**

In this variant the conflict set is computed once at the start of a cycle and **every** rule in it is
fired (a synchronous / parallel update). Rules whose conclusion has already been scheduled in the
same cycle are skipped, so a fact such as `mammal` is asserted only once even though R1 and R2 both
conclude it.

In [29]:
def forward_chain_all(rules, initial_facts, verbose=True):
    # Variant: fires ALL applicable rules of the conflict set in each cycle.
    wm = set(initial_facts)
    firing_sequence = []
    cycle = 0

    if verbose:
        print("Observed features (initial WM):", sorted(wm))
        print("-" * 78)

    while True:
        cycle += 1
        conflict_set = [r for r in rules if r.is_applicable(wm)]

        if not conflict_set:
            if verbose:
                print("Cycle {}: conflict set = []  -> no rule can fire, halt".format(cycle))
            break

        fired_now, new_facts = [], set()
        for r in conflict_set:
            if r.conclusion in new_facts:      # already asserted in this same cycle
                continue
            new_facts.add(r.conclusion)
            fired_now.append(r.rid)

        wm |= new_facts
        firing_sequence.extend(fired_now)

        if verbose:
            print("Cycle {}: conflict set = {}".format(cycle, [r.rid for r in conflict_set]))
            print("         fire ALL -> {} , assert {}".format(fired_now, sorted(new_facts)))
            print("         WM = {}".format(sorted(wm)))
            print("-" * 78)

    return wm, firing_sequence

In [30]:
compare_facts = ["has hair", "gives milk", "eats meat", "pointed teeth",
                 "claws", "forward eyes", "tawny color", "dark spots"]

print("#" * 78)
print("# ENGINE 1 : fire FIRST applicable rule per cycle")
print("#" * 78)
wm1, seq1 = forward_chain(KB_EXT, compare_facts)
report(wm1, seq1, compare_facts)

print("\n" + "#" * 78)
print("# ENGINE 2 : fire ALL applicable rules per cycle")
print("#" * 78)
wm2, seq2 = forward_chain_all(KB_EXT, compare_facts)
report(wm2, seq2, compare_facts)

##############################################################################
# ENGINE 1 : fire FIRST applicable rule per cycle
##############################################################################
Observed features (initial WM): ['claws', 'dark spots', 'eats meat', 'forward eyes', 'gives milk', 'has hair', 'pointed teeth', 'tawny color']
------------------------------------------------------------------------------
Cycle 1: conflict set = ['R1', 'R2']
         fire R1 -> assert 'mammal'
         WM = ['claws', 'dark spots', 'eats meat', 'forward eyes', 'gives milk', 'has hair', 'mammal', 'pointed teeth', 'tawny color']
------------------------------------------------------------------------------
Cycle 2: conflict set = ['R4', 'R5']
         fire R4 -> assert 'carnivore'
         WM = ['carnivore', 'claws', 'dark spots', 'eats meat', 'forward eyes', 'gives milk', 'has hair', 'mammal', 'pointed teeth', 'tawny color']
-----------------------------------------------------------

In [31]:
print("COMPARISON OF THE TWO CONTROL STRATEGIES")   
print("=" * 78)
print("{:<28}{:<26}{}".format("", "FIRST-rule engine", "ALL-rules engine"))
print("-" * 78)
print("{:<28}{:<26}{}".format("Firing sequence", " -> ".join(seq1), " -> ".join(seq2)))
print("{:<28}{:<26}{}".format("Number of rules fired", len(seq1), len(seq2)))
print("{:<28}{:<26}{}".format(
      "Identification",
", ".join(sorted(GOALS & wm1)),
", ".join(sorted(GOALS & wm2))))
print("{:<28}{:<26}{}".format("Final WM identical?", wm1 == wm2, ""))
print("=" * 78)

COMPARISON OF THE TWO CONTROL STRATEGIES
                            FIRST-rule engine         ALL-rules engine
------------------------------------------------------------------------------
Firing sequence             R1 -> R4 -> R6            R1 -> R4 -> R6
Number of rules fired       3                         3
Identification              cheetah                   cheetah
Final WM identical?         1                         


## Observations

- The engine begins with only the observed features in working memory and derives new facts cycle
  by cycle, each fired rule making further rules applicable in the next cycle.
- For the given observations the chain **R1 -> R4 -> R6** is produced: `has hair` gives the intermediate
  conclusion `mammal`, which together with `eats meat` gives `carnivore`, which with the colour
  features finally identifies the **cheetah**. Rule chaining therefore builds abstraction level by level.
- The conflict set is not fixed: it shrinks as conclusions become old facts and grows as newly
  derived facts satisfy further rules. Requiring the conclusion to be new prevents repeated firing
  and guarantees termination.
- When the facts satisfy no identification rule (Exercise 2), the engine halts with an empty conflict
  set and reports the most specific class it could establish (`carnivore`, `bird`) or reports that
  nothing could be derived.
- Firing **all** applicable rules per cycle (Exercise 3) reaches the same final working memory but in
  fewer cycles and with a different firing sequence, because independent rules such as R1/R2 and
  R4/R5 are fired together instead of one at a time. The set of conclusions is unaffected by the
  conflict-resolution strategy; only the order and the length of the explanation chain change.

## Conclusion

A forward chaining (data-driven) inference engine was implemented and used to identify an animal
from observed features. The recorded firing sequence forms the explanation chain that will be used
by the explanation facility in Experiment 7.